# VetScribe — figure regeneration notebook

Run all cells to regenerate every figure used in the paper into `./figures/`.
Each figure is in its own cell so you can edit data, colours, or labels and re-run just that cell.

**Requirements:** `pip install matplotlib pillow`  (and `reportlab` only for the example report).


In [ ]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle

os.makedirs("figures", exist_ok=True)

# ---- shared palette + style (edit here to restyle every figure) ----
TEAL   = "#1D9E75"
TEAL_D = "#0F6E56"
TEAL_L = "#9FE1CB"
AMBER  = "#BA7517"
GREY   = "#888780"
PURPLE = "#534AB7"
plt.rcParams.update({
    "font.family": "serif", "font.size": 11, "axes.titlesize": 12,
    "axes.titleweight": "bold", "savefig.dpi": 150,
    "savefig.bbox": "tight", "figure.facecolor": "white",
})
print("setup done")


## Figure 1 — Clinician time allocation (Sinsky et al., 2016)


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4))
labels = ["Direct patient\nface time", "EHR & desk\nwork", "Other\n(exam room,\nadministrative)"]
vals = [27.0, 49.2, 23.8]                 # <-- edit these values
cols = [TEAL, AMBER, TEAL_L]
b = ax.bar(labels, vals, color=cols, edgecolor="white")
for r, v in zip(b, vals):
    ax.text(r.get_x()+r.get_width()/2, v+1, f"{v:.1f}%", ha="center", fontweight="bold")
ax.set_ylabel("Share of office-day time (%)"); ax.set_ylim(0, 60)
ax.set_title("Figure 1. Where the clinical day goes: nearly half on documentation\n(time-and-motion data, Sinsky et al., 2016)")
ax.spines[["top", "right"]].set_visible(False)
plt.savefig("figures/fig1_clinician_time_allocation.png"); plt.show()


## Figure 2 — VetScribe pipeline architecture


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 2.8)); ax.set_xlim(0, 10); ax.set_ylim(0, 3); ax.axis("off")
stages = [("Record /\nUpload", "mic + file", "#E1F5EE"),
          ("Transcribe", "gpt-4o-\ntranscribe", TEAL_L),
          ("Diarize", "gpt-5.4-mini\n(index labels)", TEAL_L),
          ("Generate", "SOAP gpt-5.4\nsummary/NER mini\nflags gpt-5.4", TEAL_L),
          ("Export", "SOAP + PDF", "#E1F5EE")]
w, gap, x = 1.7, 0.25, 0.15
for i, (t, s, c) in enumerate(stages):
    ax.add_patch(FancyBboxPatch((x, 0.7), w, 1.5, boxstyle="round,pad=0.05,rounding_size=0.12",
                                linewidth=1.2, edgecolor=TEAL_D, facecolor=c))
    ax.text(x+w/2, 1.78, t, ha="center", va="center", fontweight="bold", fontsize=10, color=TEAL_D)
    ax.text(x+w/2, 1.18, s, ha="center", va="center", fontsize=7.5, color="#222")
    if i < len(stages)-1:
        ax.add_patch(FancyArrowPatch((x+w, 1.45), (x+w+gap, 1.45), arrowstyle="-|>",
                                     mutation_scale=14, color=GREY, lw=1.4))
    x += w + gap
ax.text(5, 2.6, "Figure 2. VetScribe pipeline (OpenAI-only build) with per-stage model assignment",
        ha="center", fontweight="bold", fontsize=11)
plt.savefig("figures/fig2_vetscribe_pipeline_architecture.png"); plt.show()


## Figure 3 — Per-consult cost breakdown


In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4))
stg  = ["Transcription\n(gpt-4o-transcribe)", "SOAP\n(gpt-5.4)", "Research flags\n(gpt-5.4)",
        "Entities\n(mini)", "Diarize\n(mini)", "Owner summary\n(mini)"]
cost = [0.0600, 0.0153, 0.0135, 0.0035, 0.0028, 0.0026]    # <-- edit per-stage USD
cols = [AMBER, TEAL, TEAL, TEAL_L, TEAL_L, TEAL_L]
b = ax.barh(stg[::-1], cost[::-1], color=cols[::-1], edgecolor="white")
for r, v in zip(b, cost[::-1]):
    ax.text(v+0.001, r.get_y()+r.get_height()/2, f"${v:.4f}", va="center", fontsize=9)
ax.set_xlabel("USD per ~10-minute consultation"); ax.set_xlim(0, 0.07)
ax.set_title(f"Figure 3. Per-consult API cost by stage (total approx ${sum(cost):.3f})\nTranscription dominates at {100*cost[0]/sum(cost):.0f}% of spend")
ax.spines[["top", "right"]].set_visible(False)
plt.savefig("figures/fig3_per_consult_cost_breakdown.png"); plt.show()


## Figure 4 — Cost by model strategy


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
strat = ["Legacy\n(all gpt-4o)", "Tiered\n(recommended)", "Budget\n(mini transcribe)"]
tot   = [0.108, 0.098, 0.068]            # <-- edit totals
cols  = [GREY, TEAL, TEAL_L]
b = ax.bar(strat, tot, color=cols, edgecolor="white")
for r, v in zip(b, tot):
    ax.text(r.get_x()+r.get_width()/2, v+0.002, f"${v:.3f}", ha="center", fontweight="bold")
ax.set_ylabel("USD per consult"); ax.set_ylim(0, 0.13)
ax.set_title("Figure 4. Per-consult cost by model strategy\nTiering trims spend while upgrading SOAP/flags to gpt-5.4")
ax.spines[["top", "right"]].set_visible(False)
plt.savefig("figures/fig4_cost_by_model_strategy.png"); plt.show()


## Figure 5 — Projected annual economics (edit the assumptions!)


In [ ]:
import matplotlib.ticker as mt
consults = np.arange(8, 33, 1)
days, save_min, rate, api = 250, 3.5, 100.0, 0.10   # <-- assumptions: days/yr, min saved, $/hr, $/consult
hours = consults*days*save_min/60.0
value = hours*rate
apic  = consults*days*api
fig, ax = plt.subplots(figsize=(7, 4.3))
ax.plot(consults, value, color=TEAL, lw=2.4, label="Value of clinician time saved / yr (@ $100/hr)")
ax.plot(consults, apic, color=AMBER, lw=2.2, ls="--", label="API cost / yr")
ax.fill_between(consults, apic, value, color=TEAL, alpha=0.08)
i = list(consults).index(20)
ax.annotate(f"At 20 consults/day:\n~{hours[i]:.0f} hrs (~${value[i]:,.0f}) saved\nvs ~${apic[i]:,.0f} API cost",
            xy=(20, value[i]), xytext=(9.5, value[i]*1.02), fontsize=8.5, va="top",
            bbox=dict(boxstyle="round,pad=0.3", fc="#E1F5EE", ec=TEAL_D, lw=0.8))
ax.set_xlabel("Consultations per clinician per day"); ax.set_ylabel("US dollars per year")
ax.set_title("Figure 5. Modeled annual economics per clinician: clinician-time value\nsaved vs API cost (projection, not measured)")
ax.legend(loc="upper left", fontsize=8.5, frameon=False)
ax.spines[["top", "right"]].set_visible(False); ax.set_ylim(0, value.max()*1.12)
ax.yaxis.set_major_formatter(mt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k" if x >= 1000 else f"${x:.0f}"))
plt.savefig("figures/fig5_projected_annual_savings.png"); plt.show()


## Interface figures (Figures 6 & 7)

These are faithful matplotlib renderings of the VetScribe UI (the live app is built with Gradio).
The full drawing code is long; it is provided in `ui_figures.py` alongside this notebook. Import and call it here:


In [ ]:
# The two UI mockups are generated by helper functions kept in ui_figures.py
# (capture view -> figures/ui_capture_view.png, review view -> figures/ui_review_view.png).
try:
    import ui_figures
    ui_figures.render_capture("figures/ui_capture_view.png")
    ui_figures.render_review("figures/ui_review_view.png")
    print("UI figures regenerated")
except ImportError:
    print("Place ui_figures.py next to this notebook to regenerate the UI figures.")


## Example report (Figure 8)

Figure 8 is a real PDF produced by the app's own `pdf_export.build_pdf(...)` on a synthetic case, then
rasterised. Regenerate it from the project root with:

```python
import pdf_export, json
d = json.load(open('sessions/<session_id>/results.json'))   # any saved case
pdf_export.build_pdf(d, 'figures/example_report.pdf')
```

Then convert page 1 to PNG, e.g. `pdftoppm -png -r 130 figures/example_report.pdf figures/example_report`.
